<p style ="text-align:center">
    <img src="https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Images/BannerUFPR.png" width="700" alt="PPGOLD/PPGMNE Python:INTRO"  />
</p>


# Liga Data Science
## Prof. Eduardo Pécora

# Análise de Componentes Principais (PCA) · 1. A intuição
Tempo estimado: **15** minutos

-----

## Objetivos

Após completar esta aula, você será capaz de:

* Explicar a PCA como um giro de eixos que concentra a variância
* Encontrar a direção de maior variância por força bruta e conferir com o scikit-learn
* Interpretar cargas, scores e variância explicada

Antes de usar a PCA em dados reais, vamos entender **o que ela faz**, com um exemplo pequeno: as notas de uma turma em duas provas.

| Tarefa | Pergunta | Tempo sugerido |
|---|---|---|
| **1** | Como as duas notas se comportam? | ~3 min |
| **2–3** | Qual nota combinada diferencia mais os alunos? | ~7 min |
| **4** | O scikit-learn encontra a mesma resposta? | ~5 min |

Sequência da aula:
1. **`PPGOLD_DS_9_1_PCA_Intuicao.ipynb`** — este notebook (notas da turma)
2. `PPGOLD_DS_9_2_PCA_Atlas.ipynb` — padronização e interpretação (indicadores dos estados)
3. `PPGOLD_DS_9_3_PCA_Modelos.ipynb` — PCA dentro de um modelo preditivo

### Como trabalhar
- As células com **Tarefa N** são suas: leia o enunciado e escreva o código na célula logo abaixo.
- As demais células já estão prontas: apenas execute.
- Onde aparece **Sua resposta**, escreva em uma ou duas frases o que você concluiu.

#### Começamos importando as bibliotecas e fixando a semente aleatória

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.decomposition import PCA

np.random.seed(31415)
sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

---
# Qual nota combinada diferencia mais os alunos?

Uma turma de **200 alunos** fez duas provas: **Matemática** e **Estatística**, ambas de 0 a 10.

A coordenação quer **um único número** por aluno para montar um ranking. Uma ideia é combinar as duas notas com pesos:

$$\text{nota combinada} = w_1 \cdot \text{Matemática} + w_2 \cdot \text{Estatística}$$

**Quais pesos escolher?** Uma nota combinada só serve para ranquear se ela **diferencia** os alunos. Se todos ficassem com notas parecidas, não daria para separar ninguém. Por isso vamos procurar os pesos que deixam a nota combinada **o mais espalhada possível**, ou seja, com a **maior variância**.

É exatamente isso que a PCA faz.

#### Gerando as notas da turma (simuladas)

In [ ]:
n = 200
habilidade = np.random.normal(0, 1, n)          # "talento" comum às duas provas
mat = 6.5 + 1.8 * habilidade + np.random.normal(0, 0.8, n)
est = 6.0 + 1.1 * habilidade + np.random.normal(0, 0.6, n)

notas = pd.DataFrame({"matematica": mat.clip(0, 10),
                      "estatistica": est.clip(0, 10)})
notas.head()

### Tarefa 1 — Conhecendo os dados
1. Calcule a **variância** de cada prova (`.var()`).
2. Calcule a **correlação** entre as duas provas (`.corr()`).
3. Faça um **gráfico de dispersão** de Matemática (eixo x) × Estatística (eixo y).

Qual prova espalha mais os alunos? As notas andam juntas?

In [ ]:
# Tarefa 1 — seu código aqui

**Sua resposta:** _escreva aqui_

#### Centralizando os dados

A PCA olha apenas para o **espalhamento em torno da média**. Por isso o primeiro passo é subtrair a média de cada coluna: a nuvem de pontos passa a ficar centrada na origem (0, 0).

In [ ]:
centrado = notas - notas.mean()
centrado.mean().round(10)   # agora as médias são zero

#### Pesos = uma direção

Os pesos precisam obedecer a uma regra: $w_1^2 + w_2^2 = 1$.

> Sem essa regra seria fácil "trapacear": multiplicando os pesos por 100, a variância fica 10.000 vezes maior sem nada de novo acontecer.

Todo par de pesos que obedece a essa regra corresponde a um **ângulo** $\theta$: $w_1 = \cos\theta$ e $w_2 = \sin\theta$.

| Ângulo | Pesos (Mat, Est) | Significado |
|---|---|---|
| 0° | (1, 0) | só Matemática |
| 45° | (0,71; 0,71) | as duas com o mesmo peso |
| 90° | (0, 1) | só Estatística |

A função abaixo calcula a nota combinada para qualquer ângulo:

In [ ]:
def nota_combinada(dados, angulo_graus):
    # Projeta cada aluno na direção dada pelo ângulo (em graus).
    t = np.deg2rad(angulo_graus)
    pesos = np.array([np.cos(t), np.sin(t)])
    return dados.values @ pesos      # w1*mat + w2*est, para cada aluno

nota_combinada(centrado, 45)[:5]

### Tarefa 2 — Testando três direções
Calcule a **variância** da nota combinada para os ângulos **0°, 45° e 90°**.

Dica: `np.var(valores, ddof=1)`.

Qual dos três ângulos espalha mais os alunos? Algum deles supera a variância de Matemática sozinha?

In [ ]:
# Tarefa 2 — seu código aqui

**Sua resposta:** _escreva aqui_

### Tarefa 3 — Procurando a melhor direção "na força bruta"
1. Para cada ângulo de **0° a 179°** (de 1 em 1), calcule a variância da nota combinada.
2. Faça um gráfico de **variância × ângulo**.
3. Encontre o ângulo de **maior variância** e guarde-o na variável `theta_max`.
4. Mostre os **pesos** correspondentes: `np.cos` e `np.sin` desse ângulo (em radianos).

Dica: `np.argmax` devolve a posição do maior valor de uma lista.

In [ ]:
# Tarefa 3 — seu código aqui
angulos = np.arange(0, 180)

theta_max = ...

**Sua resposta:** _escreva aqui_

#### Vendo a direção encontrada

O gráfico abaixo usa a sua variável `theta_max`. A **seta azul** é a direção de maior espalhamento. A **seta laranja** é a direção perpendicular a ela (θ + 90°): o que "sobrou" de variação.

In [ ]:
def seta(angulo, comprimento, cor, rotulo):
    t = np.deg2rad(angulo)
    plt.arrow(0, 0, comprimento * np.cos(t), comprimento * np.sin(t),
              width=0.06, color=cor, length_includes_head=True)
    plt.text(1.12 * comprimento * np.cos(t), 1.12 * comprimento * np.sin(t), rotulo,
             color=cor, fontsize=12, fontweight="bold", ha="center")

try:
    pronto = theta_max is not ...
except NameError:
    pronto = False

if not pronto:
    print("Complete a Tarefa 3 e guarde o melhor ângulo na variável theta_max antes de rodar esta célula.")
else:
    plt.figure(figsize=(7, 7))
    plt.scatter(centrado["matematica"], centrado["estatistica"], s=20, alpha=0.5, color="grey")
    seta(theta_max, 4.5, "#0072B2", "PC1")
    seta(theta_max + 90, 2.0, "#D55E00", "PC2")
    plt.axhline(0, color="black", lw=0.5); plt.axvline(0, color="black", lw=0.5)
    plt.xlabel("Matemática (centrada)"); plt.ylabel("Estatística (centrada)")
    plt.title("As componentes principais são os novos eixos")
    plt.axis("equal")
    plt.show()

**Você acabou de encontrar a 1ª Componente Principal (PC1) na força bruta.**

- **PC1** é a direção de maior variância.
- **PC2** é perpendicular à PC1 e fica com a variação restante.
- Os **pesos** de cada variável em uma componente se chamam **cargas** (*loadings*).

Com 2 variáveis bastou testar 180 ângulos. Com 12 ou 30 variáveis, testar todas as direções seria impossível. A biblioteca `scikit-learn` resolve isso com álgebra linear, instantaneamente.

### Tarefa 4 — Conferindo com o scikit-learn
1. Ajuste `PCA(n_components=2)` aos dados **centrados**.
2. Compare `pca_notas.components_[0]` (as cargas da PC1) com os seus pesos da Tarefa 3.
3. Compare `pca_notas.explained_variance_[0]` com a variância máxima que você encontrou.
4. Verifique: a **soma das variâncias** de PC1 e PC2 é igual à soma das variâncias de Matemática e Estatística?
5. Veja `pca_notas.explained_variance_ratio_`: que fração da informação a PC1 sozinha guarda?

In [ ]:
# Tarefa 4 — seu código aqui
pca_notas = PCA(n_components=2)

**Sua resposta:** _escreva aqui_

> ### Resumo
> - PCA = **girar os eixos** para que o primeiro aponte na direção de maior espalhamento.
> - Cada componente é uma **combinação linear** das variáveis originais; os pesos são as **cargas**.
> - As componentes são **perpendiculares** entre si e ordenadas da maior para a menor variância.
> - A variância total **não muda**; ela só é concentrada nas primeiras componentes.
> - O **sinal** de uma componente é arbitrário.
>
> *Para quem quiser o nome técnico: as direções das componentes são os autovetores da matriz de covariância, e as variâncias são os autovalores. Não é preciso saber calculá-los: o `PCA()` faz isso por nós.*

---
**Próximo passo:** no notebook `PPGOLD_DS_9_2_PCA_Atlas.ipynb` aplicamos a mesma ideia a **12 indicadores sociais dos 27 estados brasileiros**, e descobrimos por que é preciso padronizar.

## Fique Conectado

- [![YouTube](https://img.icons8.com/ios-filled/40/000000/youtube-play.png)](https://www.youtube.com/@LigaDataScience/videos)  
  Explore nossos vídeos educacionais e webinars sobre ciência de dados, machine learning e inteligência artificial. Inscreva-se para não perder nenhuma atualização!

- [![LinkedIn](https://img.icons8.com/ios-filled/40/000000/linkedin.png)](https://www.linkedin.com/company/liga-data-science-ufpr/)  
  Siga-nos no LinkedIn para as últimas novidades, oportunidades de carreira e networking profissional no campo da ciência de dados.

- [![Instagram](https://img.icons8.com/ios-filled/40/000000/instagram-new.png)](https://www.instagram.com/ligadatascience/)  
  Confira nosso Instagram para conteúdos dos bastidores, destaques de eventos e o dia a dia da Liga Data Science. Faça parte da nossa jornada!

_______

## Autor

<a href="https://www.linkedin.com/in/eduardopecora/" target="_blank">Eduardo Pecora</a>

## Referências

* <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html">Scikit-learn: PCA</a>
* JAMES, G. et al. *An Introduction to Statistical Learning*, cap. 12 (Unsupervised Learning)

## Log de modificações

| Data | Versão | Modificado por | Descrição |
| ----------------- | ------- | ---------- | ---------------------------------- |
| 24-10-2021       | 1.0     | Eduardo Pecora | Criação (aula única de PCA) |
| 09-11-2026       | 2.0     | Claude (a pedido de Eduardo Pécora) | Notebook próprio: intuição geométrica com as notas da turma, tarefas passo a passo e cabeçalho/rodapé LDS |

<hr>

## <h3 align="center"> (c) LDS / UFPR 2026. All rights reserved. <h3/>